URL index: [Home](https://zzz.bwh.harvard.edu/luna-walkthrough/) | [Data](https://zzz.bwh.harvard.edu/luna-walkthrough/data/) | [S1. File QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p1/) | [S2. Signal QC](https://zzz.bwh.harvard.edu/luna-walkthrough/p2) | [S3. Staging](https://zzz.bwh.harvard.edu/luna-walkthrough/p3) | [S4. Artifacts](https://zzz.bwh.harvard.edu/luna-walkthrough/p4) | [S5. Analysis](https://zzz.bwh.harvard.edu/luna-walkthrough/p5)

Notebook index: [Index](../00_index.ipynb) | [S1. File QC](../p1/00_index.ipynb) | [S2. Signal QC](../p2/00_index.ipynb) | [S3. Staging](../p3/00_index.ipynb) | [S4. Artifacts](../p4/00_index.ipynb) | [S5. Analysis](../p5/00_index.ipynb)

---

# 4.2. EEG interpolation

Walkthrough URL = [https://zzz.bwh.harvard.edu/luna-walkthrough/p4/interp/](https://zzz.bwh.harvard.edu/luna-walkthrough/p4/interp/)


In [1]:
import lunapi as lp
proj = lp.proj()
proj.sample_list( '../harm2.lst' ) 

initiated lunapi v1.2.3 <lunapi.lunapi0.luna object at 0x1093234b0> 

read 20 individuals from ../harm2.lst


## Forcing specific channels

When we know, a priori or based on other analyses, that a channel is bad, we can instruct the interpolation method to interpolate it no matter what. To aid this step, given the observations made in prior QC steps, we've created a file `../work/data/auxiliary/badchs.txt` which is a vars (individual-variable) format file that defines a variable `${badchs}` for each individual, i.e. reflecting the above three observations.  You can view this text file by navigating to it in the file browser.

## Running interpolation

This step performs of series of scans looking for outliers -- either by epoch or by channel -- followed by interpolation epoch-by-epoch. In some cases entire channels are dropped for all epochs. The core command is `INTERPOLATE`. By default, it uses internal (64-channel) sensor map. If you have more or differently-positioned sensors, you can directly attach channel locations via the `CLOCS` command first.

__This step will take several minutes to complete.__  Will it runs, you can check the contents of `../work/clean` (it will create this folder when writing the first EDF, if it doesn't already exist).  After all 20 new EDFs are generated, it will stop.  As well as the new set of EDFs in `../work/clean`, it will output epoch-level Hjorth statistics pre- and post-interpolation.

In [5]:
#luna harm2.lst vars=luna-grins/auxiliary/badchs.txt \
# -o int-out.db \
# -s ' SIGNALS drop=A1,A2
#      TAG INTERPOLATE/0
#      SIGSTATS epoch
#      TAG  .
#      CHEP-MASK ch-th=3
#      CHEP bad-channels=${badch} channels=0.3
#      CHEP-MASK ch-th=2
#      CHEP bad-channels=${badch} dump
#      INTERPOLATE
#      TAG INTERPOLATE/1
#      SIGSTATS epoch
#      WRITE edf-dir=work/clean '

# this defines ${badch} per individual
proj.var( 'vars' , '../luna-grins/auxiliary/badchs.txt' ) 

# run interpolation on all (takes a while) 
res = proj.silent_proc( ''' 
      SIGNALS drop=A1,A2
      TAG INTERPOLATE/0
      SIGSTATS epoch
      TAG  .
      CHEP-MASK ch-th=3
      CHEP bad-channels=${badch} channels=0.3
      CHEP-MASK ch-th=2
      CHEP bad-channels=${badch} dump
      INTERPOLATE
      TAG INTERPOLATE/1
      SIGSTATS epoch
      WRITE edf-dir=../work/clean ''' )


To look at the basic individual statistics from interpolation:

In [6]:
# destrat int-out.db +INTERPOLATE 
proj.table( 'INTERPOLATE' )

,ID,NCHEP_INTERPOLATED,NE_INTERPOLATED,NE_MASKED,NE_NONE
0,F01,6673,843,0,0
1,F02,8733,861,0,0
2,F03,9189,870,0,0
3,F04,8443,877,0,0
4,F05,7388,896,0,0
5,F06,7927,817,0,0
6,F07,7412,944,0,0
7,F08,10434,842,0,0
8,F09,6307,779,0,0
9,F10,10564,921,0,0


We'll skip some of the summaries here -- see the [original walkthrough](https://zzz.bwh.harvard.edu/luna-walkthrough/p4/interp/#running-interpolation) which should have yielded similar results.

## Building the new project

The new set of interpolated EDF is in the folder `..work/clean/`. We'll now add the annotations to this folder also. (Again, it is not a prerequisute that all files belong in the same folder, nor it is necessarily best practice: it is simply the procedure adopted in this walkthrough.)

In [7]:
# cp work/harm2/*.annot work/clean/

import os
import shutil

src = '../work/harm2'
dst = '../work/clean'
ext = '.annot'

for file in os.listdir(src):
    if file.endswith(ext):
        shutil.copy2(os.path.join(src, file), os.path.join(dst, file))


Finally, we'll build a new clean sample list `c.lst`, which will be the basis for most of the analysis steps of this walkthrough.

In [8]:
# luna --build work/clean > c.lst
proj.build( '../work/clean' )
sl = proj.sample_list()
sl['Annotations'] = sl['Annotations'].map( lambda x: str(x).strip('{}\'') )
sl.to_csv( '../c.lst', sep="\t", index=None, header=False)

We'll also save some of the outputs of this run, as they will be used in the next section:

In [9]:
proj.strata()

,Command,Strata
0,CHEP,CH
1,CHEP,CH_E
2,CHEP,E
3,INTERPOLATE,BL
4,INTERPOLATE,CH
5,SIGSTATS,CH_E_INTERPOLATE
6,SIGSTATS,CH_INTERPOLATE
7,WRITE,INTERPOLATE


In [14]:
# hjorth parameters (2051316 rows × 7 columns)
ss = proj.table( 'SIGSTATS' , 'CH_E_INTERPOLATE' )

# CHEP mask (1025658 rows × 4 columns)
chep = proj.table( 'CHEP' , 'CH_E' ) 

# save both
import pickle
with open('../tmp/ss.pkl', 'wb') as f: pickle.dump(ss, f)
with open('../tmp/chep.pkl', 'wb') as f: pickle.dump(chep, f)    

---
We're now ready to move on to the [next section](03_hjorth.ipynb) in which we review EEG signals for potential artifacts (pre/post interpolation) based on Hjorth parameters.